In [ ]:
# Dependencies: env, OpenAI client, and Gradio for the web UI
import os
import json
import sqlite3
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr

In [ ]:
# Model IDs used for OpenAI and Anthropic API calls
MODEL_GPT = "gpt-4o-mini"
MODEL_CLAUDE = "claude-sonnet-4-5-20250929"

In [ ]:
load_dotenv(override=True)
openai_key = os.getenv('OPENAI_API_KEY')
anthropic_key = os.getenv('ANTHROPIC_API_KEY')

if openai_key:
    print(f"OpenAI API Key exists and begins {openai_key[:8]}")
else:
    print("OpenAI API Key not set")

if anthropic_key:
    print(f"Anthropic API Key exists and begins {anthropic_key[:7]}")
else:
    print("Anthropic API Key not set")


In [ ]:
# Create OpenAI and Anthropic clients (Anthropic via OpenAI-compatible base URL)
openai_client = OpenAI()
anthropic_client = OpenAI(api_key=anthropic_key, base_url="https://api.anthropic.com/v1")
print ( "clients are ready!!")

In [ ]:
# System prompt that defines the assistant's role for all model calls
system_prompt = """You are a helpful technical tutor who answers questions about 
Python code, software engineering, devops, LLMs and Cloud engineering ."""

In [ ]:
# Generator functions: stream chat completion chunks from OpenAI and Anthropic
def stream_openai(messages):
    """Stream completion chunks from OpenAI."""
    stream = openai_client.chat.completions.create(
        model=MODEL_GPT,
        messages=messages,
        stream=True,
    )
    for chunk in stream:
        if chunk.choices and chunk.choices[0].delta.content:
            yield chunk.choices[0].delta.content


def stream_anthropic(messages):
    """Stream completion chunks from Anthropic (via OpenAI-compatible API)."""
    stream = anthropic_client.chat.completions.create(
        model=MODEL_CLAUDE,
        messages=messages,
        stream=True,
    )
    for chunk in stream:
        if chunk.choices and chunk.choices[0].delta.content:
            yield chunk.choices[0].delta.content

In [ ]:

# Single source of truth: (display label, stream_fn)
MODEL_CHOICES = [
    ("OpenAI (gpt-4o-mini)", stream_openai),
    ("Anthropic (claude-sonnet-4-5-20250929)", stream_anthropic),
]


def answer(question, model_choice):
    """Stream an answer for the chosen model."""
    if not (question and question.strip()):
        yield "Please enter a question."
        return
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": question.strip()},
    ]
    choice_map = dict(MODEL_CHOICES)
    stream_fn = choice_map.get(model_choice, stream_openai)
    acc = ""
    for chunk in stream_fn(messages):
        acc += chunk
        yield acc

In [ ]:
# Gradio UI: model dropdown, question textbox, streamed Markdown output; Submit + Enter trigger answer()
with gr.Blocks(title="Technical Q&A") as demo:
    gr.Markdown("## Technical Q&A — Ask a question, pick a model, get a streamed answer")
    model_dropdown = gr.Dropdown(
        choices=["OpenAI (gpt-4o-mini)", "Anthropic (claude-sonnet-4-5-20250929)"],
        value="OpenAI (gpt-4o-mini)",
        label="Model",
    )
    question_in = gr.Textbox(
        placeholder="e.g. Explain: yield from {book.get('author') for book in books if book.get('author')}",
        label="Question",
        lines=3,
    )

    gr.Markdown("### Answer")
    answer_out = gr.Markdown(value="*Answer will appear here.*", show_copy_button=True)

    question_in.submit(
        answer,
        inputs=[question_in, model_dropdown],
        outputs=answer_out,
    )
    gr.Button("Submit").click(
        answer,
        inputs=[question_in, model_dropdown],
        outputs=answer_out,
    )

demo.launch(share=True)